# CreditRisk+ — escopo, hipóteses e notação

## A pergunta que o modelo responde

Um banco tem milhares de clientes. Sabe, em média, quantos vão dar calote por ano — e essa média já está no preço dos empréstimos. O que ele não sabe é **quão ruim pode ser um ano ruim**.

Essa é a pergunta do CreditRisk+: dada uma carteira, qual é a distribuição de probabilidade da perda total do ano? Não "quanto vou perder", mas "com que probabilidade eu perco mais do que X".

Da resposta saem três números que se confundem com frequência:

| Número | O que é | Para que serve |
|---|---|---|
| **Perda esperada (EL)** | a média da distribuição | é custo, não risco — vai para a provisão e para o preço |
| **VaR a 99%** | a perda que só é superada em 1 ano a cada 100 | dimensiona o colchão |
| **Capital econômico** | `VaR − EL` | o capital que absorve a surpresa, já descontado o que era esperado |

A intuição central é que **perda esperada não é risco**. Duas carteiras com a mesma EL podem ter caudas completamente diferentes: uma com muitos clientes pequenos e outra concentrada em poucos grandes. O modelo existe para medir essa diferença.

## O que o modelo é e o que não é

O CreditRisk+ é **paramétrico e analítico**. Ele não roda simulação de Monte Carlo: calcula a distribuição por recursão exata a partir de funções geradoras de probabilidade (PGFs). O resultado é determinístico e reprodutível bit a bit — rodar duas vezes dá exatamente o mesmo número.

Ele aproxima defaults condicionais por Poisson e representa a incerteza da intensidade de cada setor por fatores Gama.

**Não** modela causas do default, spreads de mercado, migração de rating nem LGD estocástica. Um cliente ou quebra ou não quebra; quanto se perde quando ele quebra é um valor fixo.

A distribuição calculada é exata **dentro do modelo discretizado e do domínio numérico escolhido**. Há três aproximações explícitas, todas visíveis e controláveis: Poisson para eventos raros, banding das exposições e truncamento da série infinita.

## Inputs e transformação de severidade

### Em linguagem simples

Cada cliente entra com cinco informações:

- **quanto ele deve** (exposição);
- **qual a chance de ele quebrar** neste ano (PD média);
- **quanto essa chance oscila** de ano para ano (desvio padrão da PD) — este é o parâmetro que cria o risco de cauda; se fosse zero, anos ruins e anos bons seriam quase iguais;
- **quanto se recupera** se ele quebrar;
- **a que fatores ele é sensível** (pesos setoriais) — dois clientes sensíveis ao mesmo fator quebram juntos com mais frequência do que o acaso sugeriria.

Depois, o modelo faz uma coisa que parece estranha e é puramente computacional: ele **arredonda as exposições para múltiplos de uma unidade $L$**. Em vez de trabalhar com valores como 358.475 e 1.089.819, trabalha com "2 unidades" e "6 unidades". Isso reduz drasticamente o número de estados possíveis e é o que torna a recursão viável.

Arredondar para cima inflaria a perda esperada. Por isso o modelo **compensa a PD para baixo**, na mesma proporção: a perda esperada final é exatamente a da carteira original, sem arredondamento nenhum.

### Formalmente

Para cada contraparte $A$ são necessários exposição $E_A$, PD média $p_A$, desvio padrão da PD $s_A$, recuperação $RR_A$ e pesos setoriais $\theta_{Ak}$ que somam um. A exposição líquida é

$$L_A=E_A(1-RR_A).$$

Escolhida a unidade $L$, definem-se a banda inteira $\nu_A=\lceil L_A/L\rceil$ e a EL em unidades $\varepsilon_A=p_A L_A/L$. A frequência ajustada

$$\mu_A=\frac{\varepsilon_A}{\nu_A}$$

preserva exatamente a EL original apesar do arredondamento, porque $\nu_A \mu_A = \varepsilon_A$ por construção.

### Glossário dos símbolos

| Símbolo | Nome | Em uma frase |
|---|---|---|
| $L$ | unidade de exposição | o "tamanho do tijolo" com que todas as perdas são construídas |
| $L_A$ | exposição líquida | quanto se perde se o cliente $A$ quebrar |
| $\nu_A$ | banda | a exposição de $A$ medida em tijolos, arredondada para cima |
| $p_A$ | PD média | chance anual de $A$ quebrar |
| $s_A$ | volatilidade da PD | o quanto essa chance varia entre anos |
| $\varepsilon_A$ | perda esperada em unidades | $p_A \times$ exposição, em tijolos |
| $\mu_A$ | frequência ajustada | a PD depois da compensação do arredondamento |
| $\mu_k$ | média do setor $k$ | número esperado de defaults atribuído ao fator $k$ |
| $\sigma_k$ | volatilidade do setor $k$ | o quanto esse número oscila; **é daqui que vem a cauda** |
| $\theta_{Ak}$ | peso setorial | quanto do risco de $A$ é explicado pelo fator $k$ |
| $\alpha_k,\beta_k$ | forma e escala | os dois parâmetros da Gama que representa o fator $k$ |
| $A_n$ | a resposta | probabilidade de a perda total ser exatamente $n$ tijolos |

## Estrutura probabilística

### Em linguagem simples

O modelo é construído em duas camadas.

**Camada 1 — o número de defaults.** Se cada cliente tem uma chance pequena e independente de quebrar, o número total de quebras no ano segue uma distribuição de Poisson. É o mesmo raciocínio de "quantas ligações chegam a um call center por hora".

**Camada 2 — o tamanho das perdas.** Saber que três clientes quebraram não diz quanto se perdeu: depende de *quais* três. Um cliente grande vale por vinte pequenos. O modelo compõe a contagem de defaults com a distribuição dos tamanhos de exposição. Por isso a perda **não** é Poisson mesmo quando a contagem de defaults é.

**O ingrediente que cria a cauda.** Se parasse aí, o modelo subestimaria o risco grosseiramente, porque na prática as taxas de default não são estáveis: em recessão, *todo mundo* fica mais propenso a quebrar ao mesmo tempo. O CreditRisk+ representa isso deixando a própria taxa média de cada setor ser aleatória, seguindo uma distribuição Gama.

Vale insistir num ponto que gera confusão: o fator Gama **não** é a incerteza sobre um cliente específico. É a incerteza sobre a *conjuntura* — quantos defaults o setor terá neste ano. Todos os clientes expostos àquele fator sobem e descem juntos, e é exatamente essa sincronia que engorda a cauda.

Setores diferentes são independentes entre si. Essa é uma hipótese forte e uma limitação reconhecida do modelo original.

### Formalmente

No caso fixo, a PGF de perdas é

$$G(z)=\exp\left[\sum_A \mu_A(z^{\nu_A}-1)\right].$$

No caso variável, cada setor possui uma intensidade $x_k\sim\Gamma(\alpha_k,\beta_k)$ com média $\mu_k$ e desvio $\sigma_k$. Condicionalmente aos fatores, os defaults são independentes; incondicionalmente, contrapartes que compartilham fatores tornam-se dependentes. Setores distintos são independentes na formulação original.

Misturar uma Poisson com uma Gama produz uma **binomial negativa**, cuja variância $\mu_k+\sigma_k^2$ é maior que a da Poisson ($\mu_k$). Esse excedente $\sigma_k^2$ é, literalmente, o risco sistemático.

O setor específico de A12.3 mantém sua contribuição à média, mas recebe $\sigma_{\text{specific}}=0$: ele é o limite Poisson de muitos fatores individuais que se cancelam na carteira, não uma binomial negativa individual adicional.

## Outputs e convenções

A saída principal é $A_n=P(\text{perda}=nL)$: a probabilidade de a perda total ser exatamente $n$ unidades. Dela derivam-se EL, variância, quantis e capital econômico $EC_q=VaR_q-EL$.

### Três armadilhas de interpretação

**1. O VaR é um ponto de uma grade discreta.** A perda só pode assumir valores que são múltiplos de $L$, então a CDF sobe em degraus. O VaR matemático é o primeiro ponto da grade cuja CDF atinge $q$. A planilha oficial interpola linearmente entre dois degraus — o que produz um número que a distribuição nunca pode realizar. A API usa o quantil discreto por padrão e oferece `interpolate=True` só para reproduzir o XLS.

**2. Contribuição de risco não é "a perda que este cliente causa".** É a parcela do risco *conjunto* atribuível a ele, o que inclui o quanto ele agrava a concentração. Contribuições ao desvio padrão seguem a equação 121 e são aditivas: somam exatamente $\sigma$. Contribuições ao percentil usam a aproximação da equação 102.

**3. Contribuição não é diferença finita.** Remover um cliente e recalcular o VaR dá um número diferente da contribuição dele. O próprio manual mostra isso: as contribuições das contrapartes 24 e 25 somam 19,7 milhões, mas removê-las reduz o VaR de 99% em 15,4 milhões. Contribuições são derivadas locais (alocação de Euler); remoções são efeitos finitos. As duas respondem a perguntas diferentes e nenhuma está errada.

### Roteiro dos notebooks

| Notebook | O que faz |
|---|---|
| 02 | o caso mais simples: taxas fixas, limite Poisson |
| 03 | adiciona a incerteza da taxa: mistura Gama e binomial negativa |
| 04–05 | exemplos oficiais de um setor; concentração e contribuições |
| 06 | horizonte de três anos por contrapartes virtuais |
| 07–08 | múltiplos setores, exclusivos e fracionários |
| 11 | aplicação sintética a uma carteira PF por safras |

Os notebooks 04 a 08 contêm asserções que comparam o resultado com a planilha oficial: se a matemática regredir, a célula falha. Toda hipótese que não pertence ao CreditRisk+ é identificada como cenário.

## O modelo inteiro em uma célula

Antes de abrir a matemática, vale ver o caminho completo — da carteira ao capital — na carteira oficial do Exemplo 1A. A célula abaixo mostra cada etapa da transformação descrita acima acontecendo de fato.

In [1]:
# Percurso completo do modelo na carteira oficial do Exemplo 1A.
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from creditriskplus import CreditRiskPlus, data

portfolio = data.create_example_1a_portfolio()

# Etapa 1 — a carteira crua: quanto cada cliente deve e qual seu rating.
print(f"Contrapartes: {len(portfolio)}   "
      f"Exposição total: {portfolio.exposure.sum():,.0f}")

# Etapa 2 — banding. A unidade L vira o "tijolo"; cada exposição é medida nele.
# nu é arredondado para cima, e a PD é compensada para baixo na mesma proporção,
# de modo que nu * pd_compensada continue valendo exatamente a perda esperada.
model = CreditRiskPlus()
model.set_portfolio(portfolio)
model.calculate_loss_distribution()
L = model.unit_size
banding = pd.DataFrame({
    'exposição': portfolio.exposure,
    'rating': portfolio.rating,
    'exposição/L': portfolio.exposure / L,
    'banda ν (arredondada)': np.ceil(portfolio.exposure / L).astype(int),
    'PD do rating': portfolio.mean_default_rate,
    'PD compensada': portfolio.mean_default_rate * (portfolio.exposure / L)
                     / np.ceil(portfolio.exposure / L),
})
print(f"\nUnidade L = {L:,.0f}\n")
display(banding.head(4).style.format(precision=4))

# Etapa 3 — a distribuição e sua leitura econômica.
el = model.expected_loss
var99 = model.get_percentile_loss(99)
leitura = pd.Series({
    'Perda esperada (custo, vai para provisão)': el,
    'Desvio padrão': model.loss_std,
    'VaR 99% (perda de 1 ano em 100)': var99,
    'Capital econômico = VaR - EL': var99 - el,
    'Massa de cauda não computada': model.tail_mass_upper_bound,
})
display(leitura.to_frame('valor').style.format('{:,.4g}'))

# A EL sobrevive intacta ao arredondamento: é a checagem da etapa 2.
assert abs(el - (portfolio.exposure * portfolio.mean_default_rate).sum()) < 1e-6
print(f"EL preservada pelo banding. VaR99 é {var99/el:.1f}x a perda esperada —"
      f"\né essa razão, e não a EL, que dimensiona o capital.")

Contrapartes: 25   Exposição total: 130,513,072

Unidade L = 202,389



,exposição,rating,exposição/L,banda ν (arredondada),PD do rating,PD compensada
0,358475,H,1.7712,2,0.3000,0.2657
1,1089819,H,5.3848,6,0.3000,0.2692
2,1799710,F,8.8923,9,0.1000,0.0988
3,1933116,G,9.5515,10,0.1500,0.1433


,valor
"Perda esperada (custo, vai para provisão)",1.422e+07
Desvio padrão,1.267e+07
VaR 99% (perda de 1 ano em 100),5.545e+07
Capital econômico = VaR - EL,4.123e+07
Massa de cauda não computada,3.93e-14


EL preservada pelo banding. VaR99 é 3.9x a perda esperada —
é essa razão, e não a EL, que dimensiona o capital.
